# Lab 02 — JOINs na prática (DuckDB)

**Onde roda:** 🟢 Browser (JupyterLite). Execute célula a célula.

Objetivo: juntar `pedidos` com `clientes` e resumir com `GROUP BY`.

In [ ]:
try:
    import duckdb
except ModuleNotFoundError:
    import piplite; await piplite.install('duckdb'); import duckdb
import pandas as pd

pedidos = pd.DataFrame([
    (1,'SP','eletronicos',1200.0,1),(2,'SP','livros',50.0,2),(3,'RJ','livros',30.0,1),
    (4,'MG','casa',80.0,3),(5,'SP','eletronicos',800.0,2),(6,'RJ','casa',150.0,4),
    (7,'SP','livros',45.0,1),(8,'MG','eletronicos',600.0,3),(9,'RJ','eletronicos',900.0,2),
    (10,'SP','casa',200.0,5),(11,'MG','livros',25.0,4),(12,'SP','eletronicos',1500.0,1),
    (13,'RJ','livros',60.0,5),(14,'MG','casa',120.0,3),(15,'SP','livros',40.0,2),
], columns=['id','estado','categoria','valor','cliente_id'])

clientes = pd.DataFrame([
    (1,'ana','São Paulo'),(2,'bruno','Rio de Janeiro'),(3,'caio','Belo Horizonte'),
    (4,'duda','Rio de Janeiro'),(5,'eva','São Paulo'),(6,'fabio','Curitiba'),
], columns=['id','nome','cidade'])
clientes

## 1. INNER JOIN: pedido + nome do cliente

In [ ]:
duckdb.query('''
    SELECT p.id, c.nome, p.valor
    FROM pedidos p
    JOIN clientes c ON p.cliente_id = c.id
    ORDER BY p.valor DESC
    LIMIT 5
''').to_df()

## 2. LEFT JOIN: todos os clientes (fabio não tem pedidos!)

In [ ]:
duckdb.query('''
    SELECT c.nome, COUNT(p.id) AS n_pedidos, COALESCE(SUM(p.valor), 0) AS total
    FROM clientes c
    LEFT JOIN pedidos p ON p.cliente_id = c.id
    GROUP BY c.nome
    ORDER BY total DESC
''').to_df()

## 3. Sua vez (mini-desafio)
Traga a **receita por cidade** (colunas `cidade` e `receita`), da maior para a menor, usando INNER JOIN. Verifique.

In [ ]:
resposta = duckdb.query('''
    SELECT c.cidade, SUM(p.valor) AS receita
    FROM pedidos p
    JOIN clientes c ON p.cliente_id = c.id
    GROUP BY c.cidade
    ORDER BY receita DESC
''').to_df()
resposta

In [ ]:
def verificar(df):
    try:
        d = dict(zip(df['cidade'], df['receita']))
        assert abs(d.get('São Paulo',0)-3035.0)<1e-6, 'São Paulo deveria somar 3035.'
        assert abs(d.get('Rio de Janeiro',0)-1965.0)<1e-6, 'Rio deveria somar 1965.'
        print('\u2705 Correto! JOIN + GROUP BY certinho.')
    except AssertionError as e:
        print('\u274c', e)

verificar(resposta)